# Pavement Condition Model

This model estimates the condition of a commercial parking lot **before we inspect it**, so staff know ahead of time if it will likely need a seal, a resurface or a full replacement.

It is trained on **real data** from the City of San Diego, which measured the condition of thousands of streets in 2023.

- Source: City of San Diego Open Data Portal, "Streets Repair Projects" ([data.sandiego.gov](https://data.sandiego.gov/datasets/streets-repair-projects/))
- License: Open Data Commons Public Domain Dedication and License (free to use)
- Data file: `pavement_condition_data.csv`

| Inputs (what the customer knows) | Output (what the model predicts) |
|---|---|
| Years since repaved | Condition score from 0 (failed) to 100 (new) |
| Years since sealed | |
| Street type (the end of the street name, like ST, AV, DR, ALLEY) | |

The condition score is only the **output**. The model never sees it as an input.

The condition score is called **PCI** (Pavement Condition Index).

In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Load the data

Each row is one block of street. **street_name** is the street's real name from the city's file, and **street_type** is the last word of that name. Street types with fewer than 50 streets are grouped as OTHER.

**-1** means there's no record of that repair.

In [2]:
data = pd.read_csv("pavement_condition_data.csv")

print("Streets:", len(data))
data.head()

Streets: 23054


,street_name,street_type,years_since_repaved,years_since_sealed,condition_score
0,OLD BATTLEFIELD RD,RD,-1,-1,49.13
1,PETTIGO DR,DR,-1,2,83.76
2,CALLE SALIDA DEL SOL,OTHER,34,11,61.52
3,SABINA DR,DR,-1,7,55.20
4,PARKSIDE AV,AV,5,15,36.58


In [3]:
data.describe().round(1)

,years_since_repaved,years_since_sealed,condition_score
count,23054.0,23054.0,23054.0
mean,11.5,10.9,65.5
std,15.0,13.5,25.9
min,-1.0,-1.0,0.0
25%,-1.0,1.0,44.9
50%,4.0,5.0,73.0
75%,25.0,20.0,86.1
max,58.0,66.0,100.0


## 2. Look at the pattern

Average condition score by years since the street was last repaved. Older pavement should score lower.

In [4]:
repaved = data[data["years_since_repaved"] >= 0]
age_groups = pd.cut(repaved["years_since_repaved"], bins=[0, 5, 10, 20, 30, 60])

repaved.groupby(age_groups, observed=True)["condition_score"].agg(["count", "mean"]).round(1)

,count,mean
years_since_repaved,,
"(0, 5]",1574,91.0
"(5, 10]",2102,79.4
"(10, 20]",2141,74.3
"(20, 30]",2387,65.3
"(30, 60]",4428,58.2


Average condition score by street type:

In [5]:
data.groupby("street_type")["condition_score"].agg(["count", "mean"]).round(1).sort_values("mean")

,count,mean
street_type,,
ALLEY,2165,45.2
STRANDWAY,58,60.6
RA,108,63.2
BL,786,64.8
AV,3349,65.7
DR,2956,65.7
PL,821,66.4
RD,1723,66.6
ST,6393,66.8


## 3. Turn street type into a number

The model only understands numbers, so each street type gets its own number. For example, ALLEY becomes 0 and AV becomes 1.

In [6]:
street_types = sorted(data["street_type"].unique())

type_numbers = {}
for number, street_type in enumerate(street_types):
    type_numbers[street_type] = number

print(type_numbers)

data["street_type_number"] = data["street_type"].map(type_numbers)
data.head()

{'ALLEY': 0, 'AV': 1, 'BL': 2, 'CR': 3, 'CT': 4, 'DR': 5, 'LN': 6, 'OTHER': 7, 'PL': 8, 'PT': 9, 'PY': 10, 'RA': 11, 'RD': 12, 'ST': 13, 'STRANDWAY': 14, 'TR': 15, 'WY': 16}


,street_name,street_type,years_since_repaved,years_since_sealed,condition_score,street_type_number
0,OLD BATTLEFIELD RD,RD,-1,-1,49.13,12
1,PETTIGO DR,DR,-1,2,83.76,5
2,CALLE SALIDA DEL SOL,OTHER,34,11,61.52,7
3,SABINA DR,DR,-1,7,55.20,5
4,PARKSIDE AV,AV,5,15,36.58,1


## 4. Split the data

80% of the streets are used to train the model. The other 20% are kept hidden, so we can test the model on streets it has never seen.

In [7]:
features = ["years_since_repaved", "years_since_sealed", "street_type_number"]

X = data[features]
y = data["condition_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training streets:", len(X_train))
print("Testing streets:", len(X_test))

Training streets: 18443
Testing streets: 4611


## 5. Train the model

A random forest is a group of decision trees that each make a guess, and the model averages their guesses. `min_samples_leaf=50` makes every guess come from at least 50 streets, so the trees don't memorize single streets. It also keeps the saved model file small.

In [8]:
model = RandomForestRegressor(n_estimators=100, min_samples_leaf=50, random_state=42)
model.fit(X_train, y_train)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease 

## 6. Test the model

**Mean absolute error** is how many points the guess is off by, on average.

**R²** shows how much of the difference between streets the model explains. 1.0 is perfect, and 0 is no better than always guessing the average.

To see if the model is actually useful, we compare it to a lazy guess: always guessing the average score.

In [9]:
predictions = model.predict(X_test)

model_error = mean_absolute_error(y_test, predictions)
lazy_error = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

print("Model is off by about", round(model_error, 1), "points on average")
print("Always guessing the average is off by about", round(lazy_error, 1), "points")
print("R² score:", round(r2_score(y_test, predictions), 2))

Model is off by about 13.5 points on average
Always guessing the average is off by about 22.0 points
R² score: 0.53


How much each input mattered to the model:

In [10]:
pd.Series(model.feature_importances_, index=features).round(2)

years_since_repaved    0.40
years_since_sealed     0.56
street_type_number     0.04
dtype: float64

## 7. Real inputs and outputs

Ten streets from the test data that the model never saw. The first three columns are the **inputs**. **actual_score** is what the city really measured, and **predicted_score** is the model's guess.

In [11]:
results = X_test.head(10).copy()
results.insert(0, "street_name", data.loc[results.index, "street_name"])
results["actual_score"] = y_test.head(10).values
results["predicted_score"] = predictions[:10].round(1)
results["points_off"] = (results["predicted_score"] - results["actual_score"]).abs().round(1)
results

,street_name,years_since_repaved,years_since_sealed,street_type_number,actual_score,predicted_score,points_off
6425,ALLEY,-1,-1,0,47.97,45.4,2.6
10853,32ND ST,6,33,13,58.28,78.2,19.9
4393,SKYLINE DR,21,2,5,82.98,82.4,0.6
1114,DEL SOL BL,-1,1,2,55.29,85.5,30.2
911,WESTBROOK AV,-1,24,1,15.94,31.4,15.5
19433,REED AV,-1,-1,1,21.38,43.9,22.5
52,CARMAR WY,-1,15,16,28.96,41.5,12.5
16583,COLLIER AV,38,1,1,87.20,86.9,0.3
14963,CARMEL VALLEY RD,-1,-1,12,69.04,66.6,2.4
516,BULLOCK DR,5,21,5,92.37,92.0,0.4


## 8. Try some examples

The city groups scores into ratings. This turns a score into the same rating names. (-1 means never sealed.)

In [12]:
def rating(score):
    if score >= 85:
        return "Good"
    elif score >= 70:
        return "Satisfactory"
    elif score >= 55:
        return "Fair"
    elif score >= 40:
        return "Poor"
    elif score >= 25:
        return "Very Poor"
    elif score >= 10:
        return "Serious"
    else:
        return "Failed"


# The last three were all repaved 25 years ago. The only difference is when they were last sealed.
examples = pd.DataFrame({
    "years_since_repaved": [2, 10, 25, 25, 25],
    "years_since_sealed":  [-1, -1, -1, 10, 2],
    "street_type_number":  [type_numbers["ST"]] * 5,
})

examples["predicted_score"] = model.predict(examples).round(1)
examples["rating"] = examples["predicted_score"].apply(rating)
examples

,years_since_repaved,years_since_sealed,street_type_number,predicted_score,rating
0,2,-1,13,88.2,Good
1,10,-1,13,63.0,Fair
2,25,-1,13,50.1,Poor
3,25,10,13,59.2,Fair
4,25,2,13,83.6,Satisfactory


## 9. Try your own parking lot

Run this cell and type in each detail, pressing Enter after each one. Type **-1** if it was never sealed or you don't know. For street type, type one from the list, like ST or BL.

Run it again to try another lot.

In [ ]:
print("Street types:", ", ".join(street_types))
print()

typed_lot = pd.DataFrame([{
    "years_since_repaved": int(input("Years since it was repaved: ")),
    "years_since_sealed": int(input("Years since it was sealed (-1 if never): ")),
    "street_type_number": type_numbers[input("Street type: ").strip().upper()],
}])

typed_score = model.predict(typed_lot)[0]
print()
print("Predicted condition score:", round(typed_score, 1), "out of 100")
print("Rating:", rating(typed_score))
print("This is an estimate before the inspection, not a final answer.")

## 10. Save the model

This saves the trained model so the website can use it later.

In [13]:
joblib.dump(model, "pavement_condition_model.joblib")
print("Saved pavement_condition_model.joblib")

Saved pavement_condition_model.joblib


## What this model can and can't do

- It's trained on **city streets in San Diego**, not Chicago parking lots. Chicago's freezing winters are harder on asphalt, so real scores here would probably drop faster. I couldn't find a public Chicago dataset with condition scores.
- It only knows when the pavement was repaved and sealed and what type of street it is. Traffic, weather and how well it was built also matter, and they aren't in the data.
- Street type helps a little because it hints at how a street is used: for example, alleys usually score lower.
- Streets with no repair records (-1 for both) all get about the same guess, because the model has nothing to go on.
- It's still useful before an inspection: pavement that hasn't been sealed or repaved in a long time is usually in much worse shape.